Attribution: Iker Gutierrez Fandiño.

# Fine-tuned NLLB for Spanish-to-Asturian MT

## 0. Work environment configuration
Install required libraries: `datasets`, `sacrebleu`, `evaluate`, `torchao`, `transformers`, and `accelerate`.

In [1]:
%pip install datasets sacrebleu
%pip install evaluate
%pip install -U torchao
%pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 65.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


Import core libraries for dataset handling, model training, LoRA fine-tuning, and evaluation.

In [ ]:
from datasets import load_dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    #EarlyStoppingCallback
)

from peft import LoraConfig, get_peft_model, TaskType

import numpy as np
import evaluate

In [ ]:
import transformers
print(transformers.__version__)

5.8.0


## 1. Data

Authenticate with HuggingFace Hub to access gated datasets and models.

In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Load the ES-AST training corpus from AINA and the FLORES+ dev/test splits. Subsample number of sentences for Colab efficiency: 5000 for training, 500 for validation, and 500 for test evaluation.

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import os
import json

# =========================
# TRAIN SET (AINA)
# =========================

aina = load_dataset("projecte-aina/ES-AST_Parallel_Corpus")

aina = aina.rename_columns({
    "es": "src",
    "ast": "tgt"
})

# shuffle + subsample
train_set = aina["train"].shuffle(seed=42)

# Subsample
train_set = train_set.select(range(5000))


README.md: 0.00B [00:00, ?B/s]

es-ast_corpus.parquet:   0%|          | 0.00/165M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/704378 [00:00<?, ? examples/s]

In [ ]:
# =========================
# DEV AND TEST SETS (FLORES)
# =========================

def load_flores():

    es_dev = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="dev")
    ast_dev = load_dataset("openlanguagedata/flores_plus", "ast_Latn", split="dev")

    es_test = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="devtest")
    ast_test = load_dataset("openlanguagedata/flores_plus", "ast_Latn", split="devtest")

    dev_dataset = Dataset.from_dict({
        "src": es_dev["text"],
        "tgt": ast_dev["text"]
    })

    test_dataset = Dataset.from_dict({
        "src": es_test["text"],
        "tgt": ast_test["text"]
    })

    return dev_dataset, test_dataset

dev_set, test_set = load_flores()

# Sumsample dev set
dev_set = dev_set.select(range(500))
test_set = test_set.select(range(500))

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

In [ ]:
dataset = DatasetDict({
    "train": train_set,
    "dev": dev_set,
    "test": test_set
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['src', 'tgt'],
        num_rows: 5000
    })
    dev: Dataset({
        features: ['src', 'tgt'],
        num_rows: 500
    })
    test: Dataset({
        features: ['src', 'tgt'],
        num_rows: 500
    })
})


Save the dataset in HuggingFace format (for training) and JSONL format (for human inspection) to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
BASE_PATH = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/data_small"
HF_PATH = os.path.join(BASE_PATH, "hf_dataset")
JSONL_PATH = os.path.join(BASE_PATH, "jsonl")

os.makedirs(HF_PATH, exist_ok=True)
os.makedirs(JSONL_PATH, exist_ok=True)

# =========================
# EXPERIMENT FORMAT (HF)
# =========================

dataset.save_to_disk(HF_PATH)
print(f"HF dataset saved to: {HF_PATH}")

# =========================
# HUMAN-READABLE (JSONL)
# =========================

for split in ["train", "dev", "test"]:
    path = os.path.join(JSONL_PATH, f"{split}.jsonl")

    with open(path, "w", encoding="utf-8") as f:
        for example in dataset[split]:
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

    print(f"Saved {split} JSONL to: {path}")

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

HF dataset saved to: /content/drive/MyDrive/Master LAP/MT/asturian-shared-task/data_small/hf_dataset
Saved train JSONL to: /content/drive/MyDrive/Master LAP/MT/asturian-shared-task/data_small/jsonl/train.jsonl
Saved dev JSONL to: /content/drive/MyDrive/Master LAP/MT/asturian-shared-task/data_small/jsonl/dev.jsonl
Saved test JSONL to: /content/drive/MyDrive/Master LAP/MT/asturian-shared-task/data_small/jsonl/test.jsonl


Reload the saved dataset from Google Drive as a checkpoint, allowing the notebook to resume without redownloading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from datasets import load_from_disk
import os

HF_PATH = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/data_small/hf_dataset"

dataset = load_from_disk(HF_PATH)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['src', 'tgt'],
        num_rows: 5000
    })
    dev: Dataset({
        features: ['src', 'tgt'],
        num_rows: 500
    })
    test: Dataset({
        features: ['src', 'tgt'],
        num_rows: 500
    })
})


## 2. Load model and tokenizer
Load the NLLB-200-distilled-600M base model and tokenizer. Set Spanish as source language and Asturian as target language via `forced_bos_token_id`.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Language setup
SRC_LANG = "spa_Latn"
TGT_LANG = "ast_Latn"

tokenizer.src_lang = SRC_LANG
forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

base_model.generation_config.forced_bos_token_id = forced_bos_token_id

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

## 3. Preprocessing
Tokenize all splits using the NLLB tokenizer with a maximum sequence length of 96 tokens, truncating longer sequences.

In [ ]:
max_length = 96

def preprocess(examples):
    return tokenizer(
        examples["src"],
        text_target=examples["tgt"],
        src_lang="spa_Latn",
        tgt_lang="ast_Latn",
        max_length=max_length,
        truncation=True
    )

tokenized_datasets = {
    split: ds.map(
        preprocess,
        batched=True,
        num_proc=2,
        remove_columns=ds.column_names
    )
    for split, ds in dataset.items()
}

tok_train = tokenized_datasets["train"]
tok_dev = tokenized_datasets["dev"]
tok_test = tokenized_datasets["test"]

In [ ]:
# Sanity check
print(tok_train)
print(tok_train[0])

print(tok_dev)
print(tok_dev[0])

print(tok_test)
print(tok_test[0])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5000
})
{'input_ids': [256161, 12194, 61771, 42762, 321, 363, 336, 84971, 79, 150585, 1929, 126437, 32645, 1115, 336, 199730, 5432, 629, 248079, 182264, 35, 10179, 31914, 79, 147432, 363, 96, 1888, 83931, 35, 5630, 248090, 10669, 245, 48292, 2688, 336, 16060, 212185, 79, 2456, 58, 75513, 231, 5244, 1569, 321, 119193, 710, 248105, 248137, 93855, 248075, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [256161, 12194, 61771, 67664, 1533, 321, 3810, 17175, 103966, 79, 192936, 4554, 126437, 5781, 2685, 199730, 5432, 1107, 248079, 182264, 35, 54570, 30994, 79, 147432, 363, 96, 1888, 127029, 35, 5630, 248090, 31361, 140221, 55, 248116, 25411, 138911, 79, 2456, 58, 432, 14634, 5244, 1569, 321, 119193, 710, 248105, 248137, 93855, 248075, 2]}
Dataset({
    features: ['in

## 4. Inference with baseline
Run greedy decoding with the base model on the test set.

In [ ]:
import torch
import json
import os
import warnings

# =========================
# SILENCE WARNINGS
# =========================
warnings.filterwarnings("ignore")

# =========================
# USE EXISTING MODEL (from Section 3)
# =========================

baseline_model = base_model
baseline_tokenizer = tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
baseline_model.to(device)
baseline_model.eval()

# =========================
# LANGUAGE SETTINGS
# =========================
baseline_tokenizer.src_lang = "spa_Latn"
forced_bos_token_id = baseline_tokenizer.convert_tokens_to_ids("ast_Latn")

# avoid warning
baseline_model.generation_config.max_length = None
baseline_model.generation_config.forced_bos_token_id = forced_bos_token_id

# =========================
# TOKENIZED TEST SET
# =========================
test_ds = tok_test

# =========================
# OUTPUT DIR
# =========================

SAVE_DIR = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/predictions_baseline"
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# INFERENCE
# =========================

preds = []

for i in range(0, len(test_ds), 16):
    batch = test_ds[i:i+16]

    inputs = baseline_tokenizer.pad(
        {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        },
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = baseline_model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=1,
            forced_bos_token_id=forced_bos_token_id
        )

    batch_preds = baseline_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    preds.extend(batch_preds)

    # =========================
    # PRINT PREDICTIONS (BATCH)
    # =========================
    for j, pred in enumerate(batch_preds):
        print(f"[{i + j}] {pred}")

# =========================
# SAVE PREDICTIONS
# =========================

pred_path = os.path.join(SAVE_DIR, "predictions.json")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)

print(f"\nSaved predictions to: {pred_path}")

# =========================
# SAVE REFERENCES
# =========================

refs = test_ds["labels"]

decoded_refs = [
    baseline_tokenizer.decode([t for t in r if t != -100], skip_special_tokens=True)
    for r in refs
]

ref_path = os.path.join(SAVE_DIR, "references.json")

with open(ref_path, "w", encoding="utf-8") as f:
    json.dump(decoded_refs, f, ensure_ascii=False, indent=2)

print(f"Saved references to: {ref_path}")

[0] Currently, tenemos ratones de cuatro meses d'edá que antes yeren diabéticos y que nun son más , añadió.
[1] La investigación sigue na so etapa inicial, según indicaba el Dr. Ehud Ur, profesor na carrera de medicina de la Universidá de Dalhousie, en Halifax, Nova Escocia, y director del departamentu clínicu y científicu de la Asociación Canadiense de Diabetes.
[2] Como otros especialistes, ye escépticu sobre si la diabetes tien cura y remarca que estos descubrimientos nun son relevantes pa aquellos que ya tienen diabetes tipo 1.
[3] El lunes, Sara Danius, secretaria permanente del Comité del Premiu Nobel de Literatura de la Academia Sueca, declaró públicamente nun programa de radio de Sveriges, Suecia, que, al nun poder contactar directamente con Bob Dylan pa comunicarle'l premiu en Literatura 2016, el comité había abandonáu los sos esfuerzos pa comunicase con él.
[4] Danius declaró: "Nun facemos nada agora. Llamé y envié correos electrónicos al so asistente más cercu y recibíes res

## 5. Fine-tuning

### 5.1. Low-Rank Adaptation (LoRA)
For efficient fine-tuning, configure LoRA with rank 8 and alpha 16, targeting the four attention projection layers of the model.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


### 5.2. Training
Define training hyperparameters: batch size 8, learning rate 5e-5, 1.5 epochs, loss-based validation every 100 steps, no metric-based model selection.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/model_ft",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,

    num_train_epochs=1.5,

    learning_rate=5e-5,
    warmup_steps=0,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="no",

    predict_with_generate=False,
    fp16=True,

    logging_steps=10,
    report_to="none",

    load_best_model_at_end=False,

    seed=42
)

Instantiate the data collator and Seq2SeqTrainer:

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model)

trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_dev,
    data_collator=data_collator,
    compute_metrics=None,
    callbacks=[]
    #callbacks=[EarlyStoppingCallback(early_stopping_patience=3,early_stopping_threshold=0.01)]
)

Run a quick sanity check on a single batch to verify input and label shapes.

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(tok_train, batch_size=2, collate_fn=data_collator)
batch = next(iter(loader))

print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)

print("\nlabels:")
print(batch["labels"])

input_ids shape: torch.Size([2, 65])
labels shape: torch.Size([2, 63])

labels:
tensor([[256161,  12194,  61771,  67664,   1533,    321,   3810,  17175, 103966,
             79, 192936,   4554, 126437,   5781,   2685, 199730,   5432,   1107,
         248079, 182264,     35,  54570,  30994,     79, 147432,    363,     96,
           1888, 127029,     35,   5630, 248090,  31361, 140221,     55, 248116,
          25411, 138911,     79,   2456,     58,    432,  14634,   5244,   1569,
            321, 119193,    710, 248105, 248137,  93855, 248075,      2,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100],
        [256161,   1050, 148887,    153,    340,    130,  17175, 238886,     82,
          47243, 243920,    138,   1621, 248105,   1833, 131620,   3918,    336,
         213877, 243920,    138,   1621,    956,  66151,    597,   2054,     79,
         100348,     79,   1161,  20271,   3780, 248179,     82,  37012, 135733,
             79,  63412,    

Fine-tune:

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
100,2.787614,2.206334
200,3.266887,2.202008
300,3.225328,2.173094
400,3.235880,2.161382
500,3.501775,2.165962
600,3.224741,2.167345
700,3.135902,2.163368
800,3.565999,2.163886
900,3.112112,2.163539
938,3.457077,2.163532


TrainOutput(global_step=938, training_loss=3.2836296919312304, metrics={'train_runtime': 325.7862, 'train_samples_per_second': 23.021, 'train_steps_per_second': 2.879, 'total_flos': 1090578489606144.0, 'train_loss': 3.2836296919312304, 'epoch': 1.5008})

Save the adapter weights and tokenizer to Drive:

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/model_ft"

peft_model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

('/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/model_ft/tokenizer_config.json',
 '/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/model_ft/tokenizer.json')

## 6. Inference
Load the saved LoRA adapter and run beam search decoding (4 beams) on the raw test source sentences.

In [ ]:
import torch
import json
import os
import warnings
from google.colab import drive
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

drive.mount('/content/drive', force_remount=True)

# =========================
# SILENCE WARNINGS
# =========================
warnings.filterwarnings("ignore")

# =========================
# LOAD MODEL
# =========================

MODEL_PATH = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/model_ft"

peft_model = PeftModel.from_pretrained(base_model, MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
peft_model.to(device)
peft_model.eval()

# =========================
# TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

tokenizer.src_lang = "spa_Latn"

forced_bos_token_id = tokenizer.convert_tokens_to_ids("ast_Latn")

peft_model.generation_config.max_length = None
peft_model.generation_config.forced_bos_token_id = forced_bos_token_id

# =========================
# TOKENIZER
# =========================
MODEL_PATH = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/model_ft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

tokenizer.src_lang = "spa_Latn"

forced_bos_token_id = tokenizer.convert_tokens_to_ids("ast_Latn")

peft_model.generation_config.max_length = None
peft_model.generation_config.forced_bos_token_id = forced_bos_token_id

# =========================
# RAW TEST DATA (IMPORTANT FIX)
# =========================
test_src = dataset["test"]["src"]

# =========================
# INFERENCE
# =========================
preds = []

batch_size = 8

for i in range(0, len(test_src), batch_size):
    batch_src = test_src[i:i + batch_size]

    inputs = tokenizer(
        batch_src,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=96
    ).to(device)

    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=4,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            forced_bos_token_id=forced_bos_token_id
        )

    batch_preds = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    preds.extend(batch_preds)

    # =========================
    # PRINT
    # =========================
    for j, pred in enumerate(batch_preds):
        print(f"[{i + j}] {pred}")

Mounted at /content/drive


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer NllbTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[0] La so primer apaición foi nel estrenu de la cuarta temporada de la so hestoria na serie enllargar hasta un total d'once episodios.[16] N'abandonando la serie na cuarta temporada fixo un cameo nel final de la serie al revelase la identidá de la enigmática Gossip Girl.
[1] La investigación ta asitiada na so etapa inicial, según indica'l Dr. Ehud Ur, docente na carrera de medicina de la Universidá de Dalhousie, en Halifax, Nueva Escocia, y direutor del departamentu clínicu y científicu de la Asociación Canadiense de Diabetes.
[2] Como otros especialistes, ye escépticu sobre si la diabetes tien cura y remarca qu'estos descubrimientos nun son relevantes pa los que yá padecen de diabetes de tipu 1.
[3] La so primer apaición foi nel estrenu d'una serie de películes de televisión de la so primer temporada, na cual ganó'l Premiu Nobel de la Lliteratura de la Universidá de Suecia.
[4] Danius declaró: Actualmente nun facemos nada. llamé y envié correos electrónicos a la so asistente más cerca

Save predictions and references to Drive.

In [ ]:
import os
import json
from google.colab import drive

# =========================
# DRIVE
# =========================
drive.mount('/content/drive', force_remount=True)

# =========================
# OUTPUT DIR
# =========================
SAVE_DIR = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task/predictions_ft"
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# SAVE PREDICTIONS
# =========================
pred_path = os.path.join(SAVE_DIR, "predictions.json")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)

print(f"\nSaved predictions to: {pred_path}")

# =========================
# SAVE REFERENCES
# =========================
refs = list(dataset["test"]["tgt"])

ref_path = os.path.join(SAVE_DIR, "references.json")

with open(ref_path, "w", encoding="utf-8") as f:
    json.dump(refs, f, ensure_ascii=False, indent=2)

print(f"Saved references to: {ref_path}")

Mounted at /content/drive

Saved predictions to: /content/drive/MyDrive/Master LAP/MT/asturian-shared-task/predictions_ft/predictions.json
Saved references to: /content/drive/MyDrive/Master LAP/MT/asturian-shared-task/predictions_ft/references.json


## 7. Evaluation

Remount Drive to be able to load predictions and references as a checkpoint:

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Compute BLEU and chrF++ (SacreBLEU) for both the baseline and fine-tuned model, and print a side-by-side comparison table:

In [4]:
import json
import sacrebleu
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Master LAP/MT/asturian-shared-task"

def evaluate_run(folder):
    with open(f"{folder}/predictions.json", "r", encoding="utf-8") as f:
        preds = json.load(f)

    with open(f"{folder}/references.json", "r", encoding="utf-8") as f:
        refs = json.load(f)

    # =========================
    # BLEU
    # =========================
    bleu_score = sacrebleu.corpus_bleu(
        preds,
        [refs]
    ).score

    # =========================
    # chrF++
    # =========================
    chrfpp = sacrebleu.corpus_chrf(preds, [refs], word_order=2)
    chrfpp_score = chrfpp.score

    return round(bleu_score, 2), round(chrfpp_score, 2)

# =========================
# RUN EVALUATIONS
# =========================

baseline_bleu, baseline_chrfpp = evaluate_run(f"{BASE_DIR}/predictions_baseline")
ft_bleu, ft_chrfpp = evaluate_run(f"{BASE_DIR}/predictions_ft")

# =========================
# TABLE
# =========================

results_df = pd.DataFrame({
    "Model": ["Baseline (NLLB)", "Fine-tuned (LoRA)"],
    "BLEU": [baseline_bleu, ft_bleu],
    "chrF++": [baseline_chrfpp, ft_chrfpp]
})

print("\n=== COMPARISON ===")
print(results_df.to_string(index=False))


=== COMPARISON ===
            Model  BLEU  chrF++
  Baseline (NLLB) 15.32   42.17
Fine-tuned (LoRA) 12.80   37.45


Save the metric results table in Drive:

In [5]:
results_df.to_csv(f"{BASE_DIR}/comparison_results.csv", index=False)